# 18 Exercises

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

**18-1.** In the last example of [Section 18.4](./18_4_pitfalls_of_nonlinear_programming.ipynb), try some more starting points to see if you can find an
even better locally optimal solution. What is the best solution you can find?

**18-2.** The following little model fence.mod determines the dimensions of a rectangular field of
maximum area that can be surrounded by a fence of given length:

```ampl
param fence > 0;
var Xfield >= 0;
var Yfield >= 0;
maximize Area: Xfield * Yfield;
subject to Enclose: 2*Xfield + 2*Yfield <= fence;
```

It's well known that the optimum field is a square.

(a) When we tried to solve this problem for a fence of 40 meters, with the default initial guess of
zero for the variables, we got the following result:

```ampl
ampl: solve;
MINOS 5.5: optimal solution found.

0 iterations, objective 0
ampl: display Xfield, Yfield;
Xfield = 0
Yfield = 0
```

What could explain this unexpected outcome? Try the same problem on any nonlinear solver
available to you, and report the behavior that you observe.

(b) Using a different starting point if necessary, run your solver to confirm that the optimal dimensions
for 40 meters of fence are indeed 10 × 10.

\(c) Experiment with an analogous model for determining the dimensions of a box of maximum volume
that can be wrapped by paper of a given area.

(d) Solve the same problem as in \(c), but for wrapping a cylinder rather than a box.

**18-3.** A falling object on a nameless planet has been observed to have approximately the following
heights h `j` at (mostly) one-second intervals `t` `j` :

```ampl
j  0.0  0.5   1.5  2.5   3.5  4.5   5.5  6.5   7.5  8.5   9.5  10.0
_t_____________________________________________________________________
     
h j  100    95   87    76   66    56   47    38   26    15    6     0
```

According to the laws of physics on this planet, the height of the object at any time should be given
by the formula

```ampl
h j = a 0 − a 1 t j − 1⁄2 a 2 t 2j ,
```

where a 0 is the initial height, a 1 is the initial velocity, and a 2 is the acceleration due to gravity.
But since the observations were not made exactly, there exists no choice of a 0 , a 1 , and a 2 that will
cause all of the data to fit this formula exactly. Instead, we wish to estimate these three values by
choosing them so as to minimize the "sum of squares"

```ampl
n
Σ [h j
j=1
            − (a 0 − a 1 t j − 1⁄2 a 2 t 2j ) ] 2 .
```

where `t` `j` and h `j` are the observations from the jth entry of the `table`, and n is the number of observations.
This expression measures the error between the ideal formula and the observed behavior.

(a) Create an AMPL model that minimizes the sum of squares for any number n of observations `t` `j`
and h `j` . This model should have three variables and an objective function, but no constraints.

(b) Use AMPL data statements to represent the sample observations given above, and solve the
resulting nonlinear program to determine the estimates of a 0 , a 1 , and a 2 .

**18-4.** This problem involves a very simple "traffic flow" network:

```ampl
b
                  a                                    d

                                 c
```

Traffic proceeds in the direction of the arrows, entering at intersection a, exiting at d, and passing
through b or c or both. These data values are given for the roads connecting the intersections:

```ampl
From   To   Time   Capacity  Sensitivity
  a     b    5.0     10         0.1
  a     c    1.0     30         0.9
  c     b    2.0     10         0.9
  b     d    1.0     30         0.9
  c     d    5.0     10         0.1
```

To be specific, we imagine that the times are in minutes, the capacities are in cars per minute, and
the sensitivities are in minutes per (car per minute).

The following AMPL statements can be used to represent networks of this kind:

```ampl
set inters;                  # road intersections
param EN symbolic in inters;                           # entrance to network
param EX symbolic in inters;                           # exit from network
set roads within {i in inters, j in inters: i <> EX and j <> EN};
param time {roads} > 0;
param cap {roads} > 0;
param sens {roads} > 0;
```

(a) What is the shortest path, in minutes, from the entrance to the exit of this network? Construct a
shortest path model, along the lines of [Figure 15-7](../15/15_2_other_network_models.ipynb#fig-15-7), that verifies your answer.

(b) What is the maximum traffic flow from entrance to exit, in cars per minute, that the network
can sustain? Construct a maximum flow model, along the lines of [Figure 15-6](../15/15_2_other_network_models.ipynb#fig-15-6), that verifies your
answer.
\(c) Question (a) above was concerned only with the speed of traffic, and question (b) only with the
volume of traffic. In reality, these quantities are interrelated. As the traffic volume on a road
increases from zero, the time required to travel the road also increases.

Travel time increases only moderately when there are just a few cars, but it increases very rapidly
as the traffic approaches capacity. Thus a nonlinear function is needed to model this phenomenon.
Let us define T[i,j], the travel time on road (i,j), by the following constraints:

```ampl
var X {roads} >= 0;           # cars per minute entering road (i,j)
var T {roads};                # travel time for road (i,j)
subject to Travel_Time {(i,j) in roads}:
       T[i,j] = time[i,j] + (sens[i,j]*X[i,j]) / (1-X[i,j]/cap[i,j]);
```

You can confirm that the travel time on (i,j) is close to time[i,j] when the road is lightly
used, but goes to infinity as the use approaches cap[i,j] cars per minute. The magnitude of
sens[i,j] controls the rate at which travel time increases, as more cars try to enter the road.
Suppose that we want to analyze how the network can best handle some number of cars per minute.
The objective is to minimize average travel time from entrance to exit:

```ampl
param through > 0;           # cars per minute using the network
minimize Avg_Time:
       (sum {(i,j) in roads} T[i,j] * X[i,j]) / through;
```

The nonlinear expression T[i,j] * X[i,j] is travel minutes on road (i,j) times cars per
minute entering the road — hence, the number of cars on road (i,j). The summation in the
objective thus gives the total cars in the entire system. Dividing by the number of cars per minute
getting through, we have the average number of minutes for each car.

Complete the model by adding the following:

- A constraint that total cars per minute in equals total cars per minute out at each intersection,
except the entrance and exit.
- A constraint that total cars per minute leaving the entrance equals the total per minute (represented
by through) that are using the network.
- Appropriate bounds on the variables. (The example in [Section 18.4](./18_4_pitfalls_of_nonlinear_programming.ipynb) should suggest what bounds
are needed.)

Use AMPL to show that, for the example given above, a throughput of 4.0 cars per minute is optimally
managed by sending half the cars along a→b→d and half along a→c→d, giving an average
travel time of about 8.18 minutes.

(d) By trying values of parameter through both greater than and less than 4.0, develop a graph of
minimum average travel time as a function of throughput. Also, keep a record of which travel
routes are used in the optimal solutions for different throughputs, and summarize this information
on your graph.

What is the relationship between the information in your graph and the solutions from parts (a) and (b)?

(e) The model in \(c) assumes that you can make the cars' drivers take certain routes. For example,
in the optimal solution for a throughput of 4.0, no drivers are allowed to "cut through" from c to b.
What would happen if instead all drivers could take whatever route they pleased? Observation has
shown that, in such a case, the traffic tends to reach a stable solution in which no route has a travel
time less than the average. The optimal solution for a throughput of 4.0 is not stable, since — as
you can verify — the travel time on a→c→b→d is only about 7.86 minutes; some drivers would
try to cut through if they were permitted.
To find a stable solution using AMPL, we have to add some data specifying the possible routes
from entrance to exit:

```ampl
param choices integer > 0;   # number of routes
set route {1..choices} within roads;
```

Here route is an indexed collection of sets; for each r in 1..choices, the expression
route[r] denotes a different subset of roads that together form a route from EN to EX. For our
network, choices should be 3, and the route sets should be {(a,b),(b,d)},
{(a,c),(c,d)} and {(a,c),(c,b),(b,d)}. Using these data values, the stability conditions
may be ensured by one additional collection of constraints, which say that the time to travel
any route is no less than the average travel time for all routes:

```ampl
subject to Stability {r in 1..choices}:
       sum {(i,j) in route[r]} T[i,j] >=
              (sum {(i,j) in roads} T[i,j] * X[i,j]) / through;
```

Show that, in the stable solution for a throughput of 4.0, a little more than 5% of the drivers cut
through, and the average travel time increases to about 8.27 minutes. Thus traffic would have been
faster if the road from c to b had never been built! (This phenomenon is known as Braess's paradox,
in honor of a traffic analyst who noticed that when a certain link was added to Munich's road
system, traffic seemed to get worse.)

(f) By trying throughput values both greater than and less than 4.0, develop a graph of the stable
travel time as a function of throughput. Indicate, on the graph, for which throughputs the stable
time is greater than the optimal time.

(g) Suppose now that you have been hired to analyze the possibility of opening an additional winding
road, directly from a to d, with travel time 5 minutes, capacity 10, and sensitivity 1.5. Working
with the models developed above, write an analysis of the consequences of making this change, as
a function of the throughput value.

**18-5.** Return to the model constructed in (e) of the previous exercise. This exercise asks about
reducing the number of variables by substituting some out, as explained in [Section 18.2](./18_2_nonlinear_variables.ipynb).

(a) Set the option show_stats to 1, and solve the problem. From the extra output you get, verify
that there are 10 variables.

Next repeat the session with option substout set to 1. Verify from the resulting messages that
some of the variables are eliminated by substitution. Which of the variables must these be?

(b) Rather than setting substout, you can specify that a variable is to be substituted out by placing
an appropriate = phrase in its `var` declaration. Modify your model from (a) to use this feature,
and verify that the results are the same.

\(c) There is a long expression for average travel time that appears twice in this model. Define a
new variable Avg to stand for this expression. Verify that AMPL also substitutes this variable out
when you solve the resulting model, and that the result is the same as before.

**18-6.** In Modeling and Optimization with GINO, Judith Liebman, Leon Lasdon, Linus Schrage
and Allan Waren describe the following problem in designing a steel tank to hold ammonia. The
decision variables are

```ampl
T       the temperature inside the tank
I       the thickness of the insulation
```

The pressure inside the tank is a function of the temperature,

```ampl
P = e − 3950/(T + 460 ) + 11. 86
```

It is desired to minimize the cost of the tank, which has three components: insulation cost, which
depends on the thickness; the cost of the steel vessel, which depends on the pressure; and the cost
of a recondensation process for cooling the ammonia, which depends on both temperature and insulation
thickness. A combination of engineering and economic considerations has yielded the following
formulas:

```ampl
C I = 400I 0. 9
C V = 1000 + 22 (P − 14. 7 ) 1. 2
C R = 144 ( 80 − T)/ I
```

(a) Formulate this problem in AMPL as a two-variable problem, and alternatively as a six-variable
problem in which four of the variables can be substituted out. Which formulation would you prefer
to work with?

(b) Using your preferred formulation, determine the parameters of the least-cost tank.

\(c) Increasing the factor 144 in C R causes a proportional increase in the recondensation cost. Try
several larger values, and describe in general terms how the total cost increases as a function of the
increase in this factor.

**18-7.** A social accounting matrix is a `table` that shows the flows from each sector in an economy
to each other sector. Here is simple five-sector example, with blank entries indicating flows known
to be zero:

```ampl
LAB     H1     H2    P1    P2     total
LAB            15       3   130   80      220
H1       ?                                 ?
H2       ?                                 ?
P1             15     130         20      190
P2             25      40    55           105
```

If the matrix were estimated perfectly, it would be balanced: each row sum (the flow out of a sector)
would equal the corresponding column sum (the flow in). As a practical matter, however,
there are several difficulties:

- Some entries, marked ? above, have no reliable estimates.
- In the estimated `table`, the row sums do not necessarily equal the column sums.
- We have separate estimates of the total flows into (or out of) each sector, shown to the right of
the rows in our `table`. These do not necessarily equal the sums of the estimated rows or columns.

Nonlinear programming can be used to adjust this matrix by finding the balanced matrix that is
closest, in some sense, to the one given.

For a set S of sectors, let E T ⊆S be the subset of sectors for which we have estimated total flows,
and let E A ⊆S×S contain all sector pairs (i, j) for which there are known estimates. The given
data values are:

```ampl
ti      estimated row/column sums, i ∈ E T
a ij    estimated social accounting matrix entries, (i, j) ∈ E A
```

Let S A ⊆S×S contain all row-column pairs (i, j) for which there should be entries in the matrix —
this includes entries that contain ? instead of a number. We want to determine adjusted entries A `i` `j` ,
for each (i, j) ∈S A , that are truly balanced:

```ampl
Σ j ∈S: (i, j) ∈S    A
A ij =    Σ j ∈S: ( j,i) ∈S   A
         A ji
```

for all `i` ∈S. You can think of these equations as the constraints on the variables A `i` `j` .
There is no best function for measuring "close", but one popular choice is the sum of squared differences
between the estimates and the adjusted values — for both the matrix and the row and column
sums — scaled by the estimated values. For convenience, we write the adjusted sums as
defined variables:

```ampl
Ti =   Σ j ∈S: (i, j) ∈S    A
                                A ij
```

Then the objective is to minimize

```ampl
Σ (i, j) ∈E   A
                  (a i j − A i j ) 2 / a i j +   Σ i ∈E     T
                                                                (t i − T i ) 2 / t i
```

Formulate an AMPL model for this problem, and determine an optimal adjusted matrix.

**18-8.** A network of pipes has the following layout:

```ampl
1                                  3                     5
               9                   8                10       7
2                                  4                     6
```

The circles represent joints, and the arrows are pipes. Joints 1 and 2 are sources of flow, and joint 9
is a sink or destination for flow, but flow through a pipe can be in either direction. Associated
with each joint `i` is an amount w `i` to be withdrawn from the flow at that joint, and an elevation e `i` :

```ampl
1    2     3    4   5     6     7    8    9    10
________________________________________________________
wi     0    0   200    0   0   200   150    0    0   150
 ei   50   40    20   20   0     0     0   20   20    20
```

Our decision variables are the flows F `i` `j` through the pipes from `i` to j, with a positive value representing
flow in the direction of the arrow, and a negative value representing flow in the opposite
direction. Naturally, flow in must equal flow out plus the amount withdrawn at every joint, except
for the sources and the sink.

The "head loss" of a pipe is a measure of the energy required to move a flow through it. In our
situation, the head loss for the pipe from `i` to `j` is proportional to the square of the flow rate:

```ampl
H i j = Kc i j F 2i j if F i j > 0,
H i j = − Kc i j F 2i j if F i j < 0,
```

where K = 4.96407 × 10 − 6 is a conversion constant, and c `i` `j` is a factor computed from the diameter,
friction, and length of the pipe:

```ampl
from      to         c ij
  1        3        6.36685
  2        4       28.8937
  3       10       28.8937
  3        5        6.36685
  3        8       43.3406
  4       10       28.8937
  4        6       28.8937
  5        6       57.7874
  5        7       43.3406
  6        7       28.8937
  8        4       28.8937
  8        9      705.251
```

For two joints `i` and `j` at the same elevation, the pressure `drop` for flow from `i` to `j` is equal to the
head loss. Both pressure and head loss are measured in feet, so that after correcting for differences
in elevation between the joints we have the relation:

```ampl
H i j = (P i + e i ) − (P j + e j )
```

Finally, we wish to maintain the pressure at both the sources and the sink at 200 feet.

(a) Formulate a general AMPL model for this situation, and put together data statements for the data
given above.

(b) There is no objective function here, but you can still employ a nonlinear solver to seek a feasible
solution. By setting the option show_stats to 1, confirm that the number of variables equals
the number of equations, so that there are no "degrees of freedom" in the solution. (This does not
guarantee that there is just one solution, however.)

Check that your solver finds a solution to the equations, and display the results.